# Lab 03: Hyperparameter Tuning with Sweep Jobs

> **Source:** Microsoft Learning &ndash; [mslearn-mlops](https://github.com/MicrosoftLearning/mslearn-mlops)
> **Original:** `experimentation/Hyperparameter tuning.ipynb` | **License:** MIT

## Learning Objectives

After completing this lab you will be able to:

1. **Define a search space** using `Choice`, `Uniform`, and `LogUniform` distributions.
2. **Configure a sweep job** with Grid, Random, or Bayesian sampling.
3. **Set resource limits** (`max_total_trials`, `max_concurrent_trials`, `timeout`).
4. **Select the best model** from sweep results in Azure ML Studio.

| | |
|---|---|
| **Estimated time** | ~20 minutes |
| **Estimated cost** | ~$0.50 |

> **Exam Tip:** The `primary_metric` you pass to `job.sweep()` must **exactly match** the metric name logged with `mlflow.log_metric()` in the training script. A mismatch causes the sweep to report no best trial.

In [ ]:
# --- Prerequisites: connect to Azure ML workspace ---
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")
except Exception:
    credential = InteractiveBrowserCredential()

ml_client = MLClient.from_config(credential=credential)
print(
    f"Connected to workspace '{ml_client.workspace_name}' "
    f"in resource group '{ml_client.resource_group_name}'"
)

## Search Space Types

When defining hyperparameters for a sweep job, you choose a **distribution** for each parameter:

| Distribution | Syntax | Use Case |
|---|---|---|
| **Choice** | `Choice(values=[0.01, 0.1, 1])` | Discrete set of values (categorical or numeric) |
| **Uniform** | `Uniform(min_value=0.0, max_value=1.0)` | Continuous range, sampled uniformly |
| **LogUniform** | `LogUniform(min_value=-6, max_value=-1)` | Log-scale continuous range (e.g., learning rates) |
| **Normal** | `Normal(mu=0, sigma=1)` | Normal distribution around a mean |
| **QUniform** | `QUniform(min_value=1, max_value=10, q=1)` | Quantised uniform (discrete steps within a range) |

> **Exam Tip -- Sampling algorithm compatibility:**
>
> | Algorithm | Compatible Distributions |
> |---|---|
> | **Grid** | `Choice` only |
> | **Random** | All distributions |
> | **Bayesian** | `Choice`, `Uniform`, `QUniform` (does **not** support `LogUniform` or `Normal`) |

## Define and Submit a Sweep Job

We reuse the `train_model_parameters.py` training script from **Lab 02** (located at `../lab02-scripts-command-jobs/scripts/`). That script:

1. Accepts `--training_data` and `--reg_rate` as arguments.
2. Trains a logistic regression model.
3. Logs `Accuracy` with `mlflow.log_metric("Accuracy", ...)`.

Below we wrap that command job with a **search space** (`Choice`) and convert it into a **sweep job** using grid sampling.

In [ ]:
from azure.ai.ml import command, Input
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.sweep import Choice

# Define the base command job with a search space for reg_rate
job = command(
    code="../lab02-scripts-command-jobs/scripts",
    command="python train_model_parameters.py --training_data ${{inputs.training_data}} --reg_rate ${{inputs.reg_rate}}",
    inputs={
        "training_data": Input(type=AssetTypes.URI_FILE, path="azureml:diabetes-data:1"),
        "reg_rate": Choice(values=[0.01, 0.1, 1]),
    },
    environment="azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
    compute="cc-ai300",
)

# Convert to a sweep job
sweep_job = job.sweep(
    sampling_algorithm="grid",
    primary_metric="Accuracy",
    goal="maximize",
)

# Set resource limits
sweep_job.set_limits(max_total_trials=3, max_concurrent_trials=2, timeout=600)

sweep_job.experiment_name = "sweep-diabetes"
sweep_job.display_name = "diabetes-sweep-grid"

# Submit the sweep job
returned_job = ml_client.jobs.create_or_update(sweep_job)
print(f"Sweep job submitted: {returned_job.name}")
print(f"Monitor at: {returned_job.studio_url}")

## Review Sweep Results

Once the sweep job completes, review results in **Azure ML Studio**:

1. Open the **Jobs** page and click on the `sweep-diabetes` experiment.
2. Click the sweep job to see the **Child runs** tab -- one run per hyperparameter combination.
3. Use the **Metrics** tab to compare Accuracy across all trials.
4. Click the best child run to inspect its logged parameters and metrics.

> **Exam Tips:**
>
> - `max_total_trials` -- upper bound on the total number of trials the sweep will run. With grid sampling and 3 `Choice` values, exactly 3 trials are created.
> - `max_concurrent_trials` -- how many trials run in parallel. Limited by available compute nodes.
> - `timeout` -- maximum wall-clock seconds for the entire sweep. If exceeded, remaining trials are cancelled.

## Key Takeaways

- **Sweep jobs** automate hyperparameter tuning by running multiple trials with different parameter values.
- **Sampling algorithms**: Grid (exhaustive), Random (sample from distributions), Bayesian (model-guided search).
- The **metric name** logged in the training script must exactly match the `primary_metric` in the sweep configuration.
- Use `set_limits()` to control cost and duration.